Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import os
import requests

import time

from pathlib import Path


This is for changing directory so that script works on your device

In [ ]:
os.chdir('Enter path of project here') # set to your project root
print(os.getcwd())

/Users/tngumah/Desktop/data_processing_ai/Final_Project/real-estate-predictions


Read in datasets

In [3]:
API_KEY = '1cd18804a068d42c5b3d5ccecdd13c508c70f197'
BASE_URL = 'https://api.census.gov/data'

VARIABLES = {
    # DP05 - Demographics
    'DP05_0001E': 'total_population',
    'DP05_0018E': 'median_age',
    'DP05_0024E': 'pop_65_plus',
    'DP05_0019E': 'pop_18_to_34',

    # DP03 - Economic
    'DP03_0062E': 'median_household_income',
    'DP03_0003E': 'employment_rate',
    'DP03_0096E': 'health_insurance_pct',

    # DP04 - Housing
    'DP04_0001E': 'total_housing_units',
    'DP04_0003E': 'vacancy_rate',
    'DP04_0046E': 'owner_occupied_pct',
    'DP04_0134E': 'median_gross_rent',
    'DP04_0089E': 'median_home_value',

    # DP02 - Social
    'DP02_0064E': 'bachelors_degree_pct',
    'DP02_0066E': 'graduate_degree_pct',
    'DP02_0001E': 'total_households',
}

In [4]:

def fetch_acs_table(table, year, variables, api_key):
    """
    Fetch specific variables from an ACS DP table
    for all zip codes in a given year.
    """
    var_string = ','.join(variables.keys())
    
    url = (
        f'{BASE_URL}/{year}/acs/acs5/profile'
        f'?get={var_string}'
        f'&for=zip%20code%20tabulation%20area:*'
        f'&key={api_key}'
    )
    
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f'  Error {response.status_code} for {table} {year}: {response.text}')
        return None
    
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns={
        'zip code tabulation area': 'zip',
        **variables
    })
    df['year'] = year
    
    return df

In [5]:
# Group variables by their source table
TABLES = {
    'DP05': {k: v for k, v in VARIABLES.items() if k.startswith('DP05')},
    'DP03': {k: v for k, v in VARIABLES.items() if k.startswith('DP03')},
    'DP04': {k: v for k, v in VARIABLES.items() if k.startswith('DP04')},
    'DP02': {k: v for k, v in VARIABLES.items() if k.startswith('DP02')},
}

YEARS = [2019, 2020, 2021, 2022, 2023]

def pull_all_tables(tables, years, api_key):
    all_dfs = []
    
    for year in years:
        print(f'\nPulling year {year}...')
        year_dfs = []
        
        for table, variables in tables.items():
            print(f'  Fetching {table}...')
            df = fetch_acs_table(table, year, variables, api_key)
            
            if df is not None:
                year_dfs.append(df[['zip', 'year'] + list(variables.values())])
            
            time.sleep(0.5)
        
        # Merge all 4 tables on zip + year
        if year_dfs:
            year_combined = year_dfs[0]
            for df in year_dfs[1:]:
                year_combined = year_combined.merge(df, on=['zip', 'year'], how='outer')
            all_dfs.append(year_combined)
    
    return pd.concat(all_dfs, ignore_index=True)


In [6]:

census_df = pull_all_tables(TABLES, YEARS, API_KEY)


Pulling year 2019...
  Fetching DP05...
  Fetching DP03...
  Fetching DP04...
  Fetching DP02...

Pulling year 2020...
  Fetching DP05...
  Fetching DP03...
  Fetching DP04...
  Fetching DP02...

Pulling year 2021...
  Fetching DP05...
  Fetching DP03...
  Fetching DP04...
  Fetching DP02...

Pulling year 2022...
  Fetching DP05...
  Fetching DP03...
  Fetching DP04...
  Fetching DP02...

Pulling year 2023...
  Fetching DP05...
  Fetching DP03...
  Fetching DP04...
  Fetching DP02...


In [15]:
print(census_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167560 entries, 0 to 167559
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   zip                      167560 non-null  object
 1   year                     167560 non-null  int64 
 2   total_population         167560 non-null  object
 3   median_age               167560 non-null  object
 4   pop_65_plus              167560 non-null  object
 5   pop_18_to_34             167560 non-null  object
 6   median_household_income  167560 non-null  object
 7   employment_rate          167560 non-null  object
 8   health_insurance_pct     167560 non-null  object
 9   total_housing_units      167560 non-null  object
 10  vacancy_rate             167560 non-null  object
 11  owner_occupied_pct       167560 non-null  object
 12  median_gross_rent        167560 non-null  object
 13  median_home_value        167560 non-null  object
 14  bachelors_degree_pct

In [7]:
def import_csvs(path_to_file):
    results = {}
    
    if Path(path_to_file).is_file() and path_to_file.endswith('.csv'):
        file_name = Path(path_to_file).stem
        results[file_name] = pd.read_csv(path_to_file)
    
    elif Path(path_to_file).is_dir():
        for file in os.listdir(path_to_file):
            nested = import_csvs(path_to_file + '/' + file)
            results.update(nested)
    
    return results

In [8]:
datasets = import_csvs('Data')

In [9]:
print(datasets.keys())

dict_keys(['Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month', 'Zip_zhvf_growth_uc_sfrcondo_tier_0.33_0.67_sm_sa_month'])


Explore and clean data

In [10]:
for name, df in datasets.items():
    print(df.info(10))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26297 entries, 0 to 26296
Data columns (total 323 columns):
 #    Column      Dtype  
---   ------      -----  
 0    RegionID    int64  
 1    SizeRank    int64  
 2    RegionName  int64  
 3    RegionType  object 
 4    StateName   object 
 5    State       object 
 6    City        object 
 7    Metro       object 
 8    CountyName  object 
 9    2000-01-31  float64
 10   2000-02-29  float64
 11   2000-03-31  float64
 12   2000-04-30  float64
 13   2000-05-31  float64
 14   2000-06-30  float64
 15   2000-07-31  float64
 16   2000-08-31  float64
 17   2000-09-30  float64
 18   2000-10-31  float64
 19   2000-11-30  float64
 20   2000-12-31  float64
 21   2001-01-31  float64
 22   2001-02-28  float64
 23   2001-03-31  float64
 24   2001-04-30  float64
 25   2001-05-31  float64
 26   2001-06-30  float64
 27   2001-07-31  float64
 28   2001-08-31  float64
 29   2001-09-30  float64
 30   2001-10-31  float64
 31   2001-11-30  float64
 32   

In [22]:
def clean_census_df(df):
    # Convert object columns to numeric
    obj_cols = ['median_age', 'median_household_income', 
                'median_gross_rent', 'median_home_value', 'pop_65_plus', 
                'pop_18_to_34', 'vacancy_rate', 'total_population', 
                'total_housing_units', 'employment_rate', 'total_housing_units', 
                'health_insurance_pct', 'bachelors_degree_pct', 'graduate_degree_pct',
                'owner_occupied_pct',]
    for col in obj_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Replace Census null sentinel
    df = df.replace(-666666666, pd.NA)
    df = df.replace(-666666666.0, pd.NA)
    
    # Convert raw counts to percentages/rates
    df['pct_65_plus']          = df['pop_65_plus'] / df['total_population']
    df['pct_18_to_34']         = df['pop_18_to_34'] / df['total_population']
    df['vacancy_rate']         = df['vacancy_rate'] / df['total_housing_units']
    df['owner_occupied_pct']   = df['owner_occupied_pct'] / df['total_housing_units']
    df['employment_rate']      = df['employment_rate'] / df['total_population']
    df['health_insurance_pct'] = df['health_insurance_pct'] / df['total_population']
    df['bachelors_degree_pct'] = df['bachelors_degree_pct'] / df['total_population']
    df['graduate_degree_pct']  = df['graduate_degree_pct'] / df['total_population']
    
    # Standardize zip as zero-padded 5-digit string
    df['zip'] = df['zip'].astype(str).str.zfill(5)
    
    # Drop raw count columns we no longer need
    df = df.drop(columns=['pop_65_plus', 'pop_18_to_34'])
    
    return df

In [23]:
census_clean = clean_census_df(census_df.copy())

In [24]:
def clean_zhvi_zip(df):
    meta_cols = [c for c in df.columns if not c.startswith('20')]
    date_cols = [c for c in df.columns if c.startswith('20')]
    
    # Standardize zip
    df['RegionName'] = df['RegionName'].astype(str).str.zfill(5)
    
    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=date_cols,
        var_name='date',
        value_name='zhvi'
    )
    
    df_long['date'] = pd.to_datetime(df_long['date'])
    df_long = df_long.dropna(subset=['zhvi'])
    df_long = df_long.sort_values(['RegionName', 'date']).reset_index(drop=True)
    
    return df_long

In [25]:
def clean_zhvf_zip(df):
    df['RegionName'] = df['RegionName'].astype(str).str.zfill(5)
    
    date_cols = [c for c in df.columns if c.startswith('20')]
    meta_cols = [c for c in df.columns if not c.startswith('20') and c != 'BaseDate']
    
    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=date_cols,
        var_name='forecast_date',
        value_name='zhvf'
    )
    
    df_long['forecast_date'] = pd.to_datetime(df_long['forecast_date'])
    
    # Summarize forecast as single growth figure per zip
    # (% change from first to last forecast date)
    forecast_summary = (
        df_long.sort_values('forecast_date')
        .groupby('RegionName')
        .apply(lambda x: (x['zhvf'].iloc[-1] - x['zhvf'].iloc[0]) / x['zhvf'].iloc[0])
        .rename('forecast_growth')
        .reset_index()
    )
    
    return forecast_summary

In [26]:
zhvi_long    = clean_zhvi_zip(datasets['Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month'].copy())
zhvf_summary = clean_zhvf_zip(datasets['Zip_zhvf_growth_uc_sfrcondo_tier_0.33_0.67_sm_sa_month'].copy())

/var/folders/dn/szlf9g8d0rn20cpcpxlsllbc0000gn/T/ipykernel_19182/840233059.py:21: RuntimeWarning: divide by zero encountered in scalar divide
  .apply(lambda x: (x['zhvf'].iloc[-1] - x['zhvf'].iloc[0]) / x['zhvf'].iloc[0])
/var/folders/dn/szlf9g8d0rn20cpcpxlsllbc0000gn/T/ipykernel_19182/840233059.py:21: RuntimeWarning: invalid value encountered in scalar divide
  .apply(lambda x: (x['zhvf'].iloc[-1] - x['zhvf'].iloc[0]) / x['zhvf'].iloc[0])


In [27]:
def build_zhvi_features(df_long):
    max_date = df_long['date'].max()
    
    def get_price_n_months_ago(n):
        cutoff = max_date - pd.DateOffset(months=n)
        return (
            df_long[df_long['date'] <= cutoff]
            .sort_values('date')
            .groupby('RegionName')['zhvi']
            .last()
        )
    
    current   = df_long.groupby('RegionName')['zhvi'].last().rename('current_zhvi')
    price_3m  = get_price_n_months_ago(3)
    price_12m = get_price_n_months_ago(12)
    price_36m = get_price_n_months_ago(36)
    price_60m = get_price_n_months_ago(60)
    
    r3  = ((current - price_3m)  / price_3m).rename('return_3m')
    r12 = ((current - price_12m) / price_12m).rename('return_12m')
    r36 = ((current - price_36m) / price_36m).rename('return_36m')
    r60 = ((current - price_60m) / price_60m).rename('return_60m')
    
    # Acceleration: annualized 3m vs 12m
    acceleration = ((r3 * 4) - r12).rename('acceleration')
    
    # Zip vs its metro median (relative value signal)
    latest = df_long[df_long['date'] == max_date].copy()
    metro_median = latest.groupby('Metro')['zhvi'].median().rename('metro_median_zhvi')
    latest = latest.join(metro_median, on='Metro')
    latest['vs_metro'] = latest['zhvi'] / latest['metro_median_zhvi']
    vs_metro = latest.set_index('RegionName')['vs_metro']
    
    # Metadata
    meta_cols = ['RegionName', 'State', 'City', 'Metro', 'CountyName', 'StateName']
    meta = df_long[meta_cols].drop_duplicates().set_index('RegionName')
    
    features = pd.concat([
        current, r3, r12, r36, r60, acceleration, vs_metro, meta
    ], axis=1).reset_index()
    
    return features

In [28]:
zhvi_features = build_zhvi_features(zhvi_long)

In [29]:
zillow_full = zhvi_features.merge(
    zhvf_summary, on='RegionName', how='left'
)

# Get most recent Census year per zip for the merge
# (we'll use the full panel later for trend features)
census_latest = census_clean[census_clean['year'] == census_clean['year'].max()]

# Merge Census onto Zillow
master_df = zillow_full.merge(
    census_latest,
    left_on='RegionName',
    right_on='zip',
    how='left'
).drop(columns='zip')

print(f'Master dataframe shape: {master_df.shape}')
print(master_df.head())

Master dataframe shape: (26297, 30)
  RegionName   current_zhvi  return_3m  return_12m  return_36m  return_60m  \
0      01001  333711.806209   0.022496    0.034000    0.173989    0.383402   
1      01002  514492.174500   0.012226    0.000336    0.147875    0.365679   
2      01005  397515.389028   0.015974    0.005123    0.189278    0.422995   
3      01007  457172.235335   0.016460    0.012722    0.159456    0.385280   
4      01008  353742.309249   0.019665    0.007749    0.155501    0.360166   

   acceleration  vs_metro State         City  ... total_housing_units  \
0      0.055982  0.933308    MA       Agawam  ...              7033.0   
1      0.048566  1.438906    MA      Amherst  ...             11089.0   
2      0.058773  0.911892    MA        Barre  ...              1876.0   
3      0.053119  1.278596    MA  Belchertown  ...              6569.0   
4      0.070912  0.989329    MA    Blandford  ...               655.0   

  vacancy_rate owner_occupied_pct  median_gross_rent  me